# 04b · Classical Models, Extra Seeds (RF, SVM, XGB) — work package WP-B

**Project:** Placement-specific axis reduction of wearable inertial sensors for gym exercise recognition (RecGym).

Notebook 01 ran RF, SVM and XGB on the 15 primary axis configurations with **seed 42 only**. RF, XGB and the SVM training sub-sample all depend on `random_state`, so one seed cannot tell how much of a macro-F1 difference is seed noise. This notebook repeats the **identical** primary grid with another seed (default **43**; run it a second time with **44**) and writes each seed to its own folder:

| Seed | Folder | Source |
|---|---|---|
| 42 | `results/raw/classical_full/` | notebook 01 (**immutable, not re-run here**) |
| 43 | `results/raw/classical_seed43/` | this notebook, `CLASSICAL_SEED=43` |
| 44 | `results/raw/classical_seed44/` | this notebook, `CLASSICAL_SEED=44` |

Separate folders are required because notebook 01's configuration hash contains the seed. The CNN seeds (42 to 46) are handled by editing notebook 02 (`tools/patch_02_cnn_seeds.py`); everything is reported in notebook `04c_seed_analysis.ipynb`.

> Nothing is reported until *you* run it. Every number comes from the real file on your Kaggle / Colab run. **Run with `SMOKE_TEST=1` first** (default), read the timing, then set `SMOKE_TEST=0`. RF, SVM and XGB need no GPU: run this notebook with the accelerator **off** and keep the weekly GPU quota for the CNN seeds.

---

## Frozen design (identical to notebook 01; **only the seed changes**)

| Decision | Value |
|---|---|
| Configurations | The 15 primary within-sensor configurations (6-axis, acc-3, gyro-3, six pairs, six singles), rebuilt and **asserted equal** to `protocol/axis_configs.json`. |
| Channels | Ax Ay Az Gx Gy Gz, device-frame channels (C1 excluded). Axes are **never** interpreted as anatomical directions. |
| Window | 4 s = 80 samples at nominal 20 Hz, 50 % overlap (step 40), never crossing a (subject, position, session, label) run boundary. Fixed a priori. |
| Windows | Rebuilt here and **asserted equal** to `protocol/windows_index.parquet` (start row, subject, position, session, label, run id). |
| Folds | Leave-one-subject-out, 10 folds, **asserted equal** to `protocol/loso_folds.json`; subject-disjointness asserted in every fold. |
| Metric | Primary: macro-F1 over the classes present in the held-out subject (12 labels incl. `Null`). Sensitivity: macro-F1 over the 11 workouts. Same secondary metrics and per-class F1 as notebook 01. |
| Features | The same 15 per-channel features, **no cross-channel features**; a k-channel configuration sees exactly 15·k features (unit-tested on all 15 configurations). |
| RF | 200 trees, `class_weight="balanced_subsample"`, `random_state` = run seed |
| SVM | RBF, C = 1, gamma = "scale", balanced weights, stratified cap of 15,000 training windows drawn with `random_state` = run seed + fold, `random_state` = run seed |
| XGB | 300 trees, depth 6, learning rate 0.1, subsample 0.8, colsample_bytree 0.8, balanced sample weights, `random_state` = run seed |
| Normalisation, class weights, SVM sub-sampling | Training subjects only, per fold |
| Tuning / early stopping | **None.** No test-subject influence by construction. |

Only `random_state` differs from notebook 01, through the single `fit_model(..., seed)` argument. The same function with `seed = 42` is used by the reproduction anchor (Section 8) to show that this notebook reproduces notebook 01's seed-42 result before any new seed is trusted.

**Deliberately not recorded here:** per-window latency (inconsistent across sessions; work package WP-D re-measures it on one machine) and the long confusion table (not needed for seed variability). Model size and node/support-vector counts are still recorded.

---

## Frozen analysis plan (executed in notebook `04c_seed_analysis`, written to `analysis_plan.json` by this notebook)

1. **Descriptive seed variability only.** For every (model, position, configuration): mean macro-F1 over the 10 held-out subjects for each seed, then the mean, SD (ddof = 1), min, max and range across seeds. No new hypothesis tests are introduced by the seed runs.
2. **The primary tests are unchanged.** The primary family (14 reduced configurations versus 6-axis per position and model, Holm within family) keeps its seed-42 p-values. Seeds are used to **re-check** the counts and the 0.05 non-inferiority claim, not to replace them.
3. **Claims re-checked per seed:** "160 of 168 reductions are Holm-significant decreases" and "only calf gyro-3 meets the 0.05 margin in all four models". The statement made in the paper is whatever the seeds support, reported with the number of seeds in which it holds.
4. **No energy claim.** Channel count is treated as the variable that sets the data to sample, send, store and process.

---

## Run control (environment variables or `RUN_OVERRIDES` in the next cell)

| Variable | Default | Meaning |
|---|---|---|
| `SMOKE_TEST` | `1` | `1`: tiny check run (3 configurations, folds 0 and 1, reduced model sizes) into `classical_seed<seed>_smoke/`. `0`: full grid into `classical_seed<seed>/`. |
| `CLASSICAL_SEED` | `43` | Seed of this run. Any integer except 42 (42 is notebook 01's run). Planned: 43 and 44. |
| `MODELS` | `XGB,RF,SVM` | Subset and order. Priority order: XGB, RF, SVM. |
| `POSITIONS` | all (`leg,pocket,wrist`) | Chunk by placement. |
| `FOLDS` | all | Comma-separated fold indices (0 to 9). |
| `CONFIG_NAMES` | all 15 | Comma-separated primary configuration names. |
| `TIME_BUDGET_MIN` | none | Stop cleanly after this many minutes, write outputs and zip, resume later. Set it 20–30 min below the session limit. |
| `RESUME_FROM` | none | Folder with a previous chunk's `results.csv`: copied in first, finished units are skipped. |
| `RESULTS_DIR` | `results/raw` | Parent of the run folder. |
| `RECGYM_CSV` | `data/RecGym.csv` | The audited data file (SHA-256 enforced). |
| `PROTOCOL_DIR` | `protocol` | Folder with `windows_index.parquet`, `loso_folds.json`, `axis_configs.json`, `windows_meta.json`. |
| `PRIMARY_DIR` | `results/raw` | Folder holding `classical_full/` and `cnn_full/` (it is also searched under `<PRIMARY_DIR>/results/raw`). Used for the reproduction anchor, the seed-vs-primary table and the window-hash check. |
| `XGB_DEVICE` | `cpu` | Notebook 01 used CPU XGBoost; keep `cpu` for comparability. |
| `REQUIRE_COMPLETE` | unset | `1`: fail at the end if any unit of this chunk is missing. |
| `FORCE_NEW_RUN` | unset | `1`: ignore a config-hash mismatch (results.csv is **not** deleted). |
| `RECGYM_SKIP_HASH` | unset | Set to skip the SHA-256 check of the data file. |

**Unit counts.** One model = 3 positions × 10 folds × 15 configurations = **450 units**. One seed with XGB + RF + SVM = **1,350 units**. Seeds 43 and 44 = **2,700 units**.

**Rough runtimes** (scaled from notebook 01's full run: RF ≈ 922 s, SVM ≈ 348 s, XGB ≈ 280 s per configuration for 3 positions × 10 folds; your hardware will differ): per seed, XGB ≈ 1.2 h, RF ≈ 3.8 h, SVM ≈ 1.5 h, total ≈ 6.5 h; both seeds ≈ 13 h.

**Chunking.** Every unit is appended to `results.csv` as soon as it finishes and finished units are skipped on rerun. Chunks run in different sessions are combined with `tools/merge_seed_chunks.py`.

## 1 · Configuration

In [ ]:
import os, sys, re, json, time, hashlib, platform, random, shutil, zipfile, warnings, itertools, io, copy
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
try:
    display
except NameError:
    display = print

T0 = time.time()

# ---- optional in-notebook overrides (handy on Kaggle / Colab, where exporting shell variables is awkward).
# Leave a key out to use the environment variable / default. Examples (add inside the dict):
#   "SMOKE_TEST": "0", "CLASSICAL_SEED": "44", "MODELS": "XGB", "POSITIONS": "wrist", "TIME_BUDGET_MIN": "600"
RUN_OVERRIDES = {"RECGYM_CSV": "/content/drive/MyDrive/Research/Gym_Sensor_Research/01_data/RecGym.csv", "PROTOCOL_DIR": "/content/drive/MyDrive/Research/AxisFit-main/protocol",
                 "PRIMARY_DIR": "/content/drive/MyDrive/Research/AxisFit-main"}
os.environ.update({str(k): str(v) for k, v in RUN_OVERRIDES.items()})

def env_list(name, default=None):
    v = os.environ.get(name)
    if v is None or not v.strip():
        return default
    return [x.strip() for x in v.split(",") if x.strip()]

# ============================ USER CONFIG ====================================
SMOKE_TEST = os.environ.get("SMOKE_TEST", "1") == "1"
SEED_PRIMARY = 42                                   # notebook 01's seed (results/raw/classical_full, immutable)
SEED = int(os.environ.get("CLASSICAL_SEED", "43"))  # seed of THIS run
SEEDS = [SEED]
DATA_PATH = os.environ.get("RECGYM_CSV", "data/RecGym.csv")
RESULTS_ROOT = Path(os.environ.get("RESULTS_DIR", "results/raw"))
PROTOCOL_DIR = Path(os.environ.get("PROTOCOL_DIR", "protocol"))
PRIMARY_DIR = Path(os.environ.get("PRIMARY_DIR", "results/raw"))
RESUME_FROM = os.environ.get("RESUME_FROM", "").strip()
EXPECTED_SHA256 = "2031c077a57e93d6b5c39da198470cb6c9b1a3c0652ab57cfc5d6a39bfa840e9"
ENFORCE_HASH = os.environ.get("RECGYM_SKIP_HASH") is None
MOUNT_DRIVE = True

FS = 20
WINDOW_SECONDS = 4
OVERLAP = 0.5

MODELS = [m.upper() for m in env_list("MODELS", ["XGB", "RF", "SVM"])]
POSITIONS = env_list("POSITIONS", None)
CONFIG_NAMES = env_list("CONFIG_NAMES", None)
FOLDS = [int(x) for x in env_list("FOLDS", [])] or None
_tb = os.environ.get("TIME_BUDGET_MIN", "").strip()
TIME_BUDGET_S = float(_tb) * 60.0 if _tb else None
XGB_DEVICE = os.environ.get("XGB_DEVICE", "cpu").lower()
FORCE_NEW_RUN = os.environ.get("FORCE_NEW_RUN") == "1"
REQUIRE_COMPLETE = os.environ.get("REQUIRE_COMPLETE") == "1"

# hyper-parameters exactly as notebook 01; the seed is passed separately (random_state) so only it changes between runs
RF_BASE = dict(n_estimators=200, max_depth=None, min_samples_leaf=1, max_features="sqrt",
               class_weight="balanced_subsample", n_jobs=-1)
SVM_BASE = dict(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", cache_size=1000)
SVM_MAX_TRAIN_WINDOWS = 15000
XGB_BASE = dict(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                tree_method="hist", n_jobs=-1)
SHUFFLE_THRESHOLD = 0.25            # label-shuffle control must stay below this macro-F1 (chance is about 0.083)
ANCHOR_TOL = 1e-3                   # classical models are deterministic for fixed library versions
# ==============================================================================

SMOKE_CONFIGS = ["6-axis", "gyro-GxGz", "single-Gz"]   # one per size class, so every code path is exercised

assert SEED != SEED_PRIMARY, "CLASSICAL_SEED must differ from 42: seed 42 is notebook 01's run and is never re-run here."
assert all(m in ("RF", "SVM", "XGB") for m in MODELS), f"unknown MODELS (this notebook runs RF, SVM, XGB only): {MODELS}"
assert len(set(MODELS)) == len(MODELS), "duplicates in MODELS"
assert XGB_DEVICE in ("cpu", "cuda"), "XGB_DEVICE must be cpu or cuda"

if SMOKE_TEST:
    RF_BASE = {**RF_BASE, "n_estimators": 30}
    XGB_BASE = {**XGB_BASE, "n_estimators": 30}
    SVM_MAX_TRAIN_WINDOWS = 4000
    FOLDS = FOLDS or [0, 1]

IN_KAGGLE = os.path.exists("/kaggle/working")
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
BASE = Path("/kaggle/working") if IN_KAGGLE else (Path("/content") if IN_COLAB else Path(".").resolve())
random.seed(SEED); np.random.seed(SEED)

if IN_COLAB and MOUNT_DRIVE and str(DATA_PATH).startswith("/content/drive") and not os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

TAG = "smoke" if SMOKE_TEST else "full"
RUN_DIR = RESULTS_ROOT / (f"classical_seed{SEED}" + ("_smoke" if SMOKE_TEST else ""))
RUN_DIR.mkdir(parents=True, exist_ok=True)

LOG = []
def log(*a):
    s = " ".join(str(x) for x in a); print(s); LOG.append(s)

def find_primary(name):
    # locate a primary-grid run folder (classical_full / cnn_full) under PRIMARY_DIR or PRIMARY_DIR/results/raw
    for b in [PRIMARY_DIR, PRIMARY_DIR / "results" / "raw"]:
        if (b / name / "results.csv").exists() or (b / name / "run_config.json").exists():
            return b / name
    return None

if RESUME_FROM:
    _src = Path(RESUME_FROM)
    assert (_src / "results.csv").exists(), f"RESUME_FROM={RESUME_FROM} has no results.csv"
    if not (RUN_DIR / "results.csv").exists():
        for _fn in ["results.csv", "run_config.json", "anchor_check.csv", "sanity_checks.json"]:
            if (_src / _fn).exists():
                shutil.copy2(_src / _fn, RUN_DIR / _fn)
        log("Resumed: copied previous chunk files from", _src)
    else:
        log("RESUME_FROM ignored: results.csv already exists in", RUN_DIR)

if XGB_DEVICE == "cuda" and "XGB" in MODELS:
    assert shutil.which("nvidia-smi") is not None, "XGB_DEVICE=cuda but no GPU is visible"

log("SMOKE_TEST =", SMOKE_TEST, "| seed:", SEED, "| run dir:", RUN_DIR, "| models:", MODELS,
    "| positions:", POSITIONS or "all", "| folds:", FOLDS or "all", "| time budget (min):", None if TIME_BUDGET_S is None else TIME_BUDGET_S / 60)
if SMOKE_TEST:
    log(">>> SMOKE TEST: configurations", SMOKE_CONFIGS, ", folds", FOLDS, ", reduced model sizes. Set SMOKE_TEST=0 for the real run. <<<")

## 2 · Axis configurations

The 15 primary configurations are rebuilt exactly as in notebooks 01/02 and compared with `protocol/axis_configs.json`. No new configuration is introduced here.

In [ ]:
IMU = ["Ax", "Ay", "Az", "Gx", "Gy", "Gz"]
acc, gyr = IMU[:3], IMU[3:]

PRIMARY_CONFIGS = {"6-axis": list(IMU), "acc-3": list(acc), "gyro-3": list(gyr)}
for a, b in itertools.combinations(acc, 2): PRIMARY_CONFIGS[f"acc-{a}{b}"] = [a, b]
for a, b in itertools.combinations(gyr, 2): PRIMARY_CONFIGS[f"gyro-{a}{b}"] = [a, b]
for k in IMU: PRIMARY_CONFIGS[f"single-{k}"] = [k]
assert len(PRIMARY_CONFIGS) == 15

assert PROTOCOL_DIR.exists(), f"Protocol folder not found: {PROTOCOL_DIR}. Set PROTOCOL_DIR to the repository's protocol/ folder."
proto_axis = json.load(open(PROTOCOL_DIR / "axis_configs.json"))
assert {k: list(v) for k, v in proto_axis["configs"].items()} == PRIMARY_CONFIGS, "primary configs differ from protocol/axis_configs.json"
proto_folds = json.load(open(PROTOCOL_DIR / "loso_folds.json"))
proto_meta = json.load(open(PROTOCOL_DIR / "windows_meta.json"))
if POSITIONS is not None:
    assert all(p in proto_meta["positions"] for p in POSITIONS), f"POSITIONS must be among {proto_meta['positions']}, got {POSITIONS}"
for _n, _ch in PRIMARY_CONFIGS.items():
    assert _ch == [c for c in IMU if c in _ch], f"{_n}: channels not in canonical order"
log("Primary configurations verified against protocol/axis_configs.json:", len(PRIMARY_CONFIGS))

if CONFIG_NAMES is None and SMOKE_TEST:
    CONFIG_NAMES = list(SMOKE_CONFIGS)
if CONFIG_NAMES is not None:
    _unknown = [c for c in CONFIG_NAMES if c not in PRIMARY_CONFIGS]
    assert not _unknown, f"CONFIG_NAMES not among the 15 primary configurations: {_unknown}"
    CONFIGS = {k: v for k, v in PRIMARY_CONFIGS.items() if k in CONFIG_NAMES}
else:
    CONFIGS = dict(PRIMARY_CONFIGS)
assert CONFIGS, "no configuration selected"
log("Configs this chunk (", len(CONFIGS), "):", list(CONFIGS))

ANALYSIS_PLAN = {
    "note": "Frozen before any extra-seed run. Statistics are computed in notebook 04c_seed_analysis, not here.",
    "work_package": "WP-B (classical seeds)",
    "seeds": {"42": "results/raw/classical_full (notebook 01, immutable)", str(SEED): f"results/raw/classical_seed{SEED}"},
    "descriptive": "per (model, position, configuration): mean macro-F1 over 10 held-out subjects per seed, then mean, SD (ddof=1), min, max, range across seeds",
    "primary_family_unchanged": "14 reduced configurations vs 6-axis per position and model, Holm within family; seed-42 p-values stay the primary ones",
    "claims_rechecked_per_seed": ["160 of 168 reductions are Holm-significant decreases", "only calf gyro-3 meets the 0.05 non-inferiority margin in all four models"],
    "only_difference_from_notebook_01": "random_state of RF, SVM, XGB and the SVM training sub-sample (seed + fold)",
    "not_recorded": ["per-window latency (WP-D)", "confusion table"],
    "energy": "no energy claim; channel count sets data to sample, send, store and process",
}
json.dump(ANALYSIS_PLAN, open(RUN_DIR / "analysis_plan.json", "w"), indent=2)

## 3 · Load data and verify it is the audited file

In [ ]:
def sha256_of(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

assert os.path.exists(DATA_PATH), f"Data file not found: {DATA_PATH}. Mount Drive / attach the dataset or set RECGYM_CSV."
DATA_SHA256 = sha256_of(DATA_PATH)
log("SHA-256:", DATA_SHA256)
if ENFORCE_HASH:
    assert DATA_SHA256 == EXPECTED_SHA256, ("The file differs from the one audited in notebook 00. "
        "Re-run the audit on this file (or update EXPECTED_SHA256 on purpose).")
    log("Hash matches the audited file.")

raw = pd.read_csv(DATA_PATH)
raw.columns = [str(c).strip() for c in raw.columns]
log("Loaded:", raw.shape, list(raw.columns))

def norm(c): return re.sub(r"[^a-z0-9]", "", str(c).lower())
ncol = {norm(c): c for c in raw.columns}
def pick(cands):
    for k in cands:
        if k in ncol: return ncol[k]
    return None
subj_col = pick(["subject", "object", "subjectid", "volunteer", "participant", "user"])
label_col = pick(["workout", "label", "activity", "exercise"])
pos_col = pick(["position", "placement", "location"])
sess_col = pick(["session", "sessionid", "day"])
chan_cols = {k: ncol.get(k.lower()[0] + "_" + k.lower()[1]) or ncol.get(k.lower()) for k in IMU}
assert subj_col and label_col and pos_col, f"Cannot detect roles in {list(raw.columns)}"
assert all(chan_cols.values()), f"Cannot detect IMU channels: {chan_cols}"
log("Roles:", dict(subject=subj_col, label=label_col, position=pos_col, session=sess_col), "| channels:", chan_cols)
if sess_col is None:
    log("WARNING: no session column; runs are delimited by (subject, position, label) only.")

subject_str = raw[subj_col].astype(str).str.strip()
subjects = sorted(subject_str.unique(), key=lambda s: (len(s), s))
labels = sorted(raw[label_col].astype(str).str.strip().unique())
positions_all = sorted(raw[pos_col].astype(str).str.strip().unique())
K = len(labels)
label_map = {l: i for i, l in enumerate(labels)}
null_ids = [label_map[l] for l in labels if "null" in l.lower()]
log("Subjects:", subjects); log("Positions:", positions_all); log("Labels:", labels, "| null ids:", null_ids)

cat = lambda s, cats: pd.Categorical(s.astype(str).str.strip(), categories=cats).codes.astype(np.int32)
subj_c = cat(raw[subj_col], subjects)
pos_c = cat(raw[pos_col], positions_all)
lab_c = cat(raw[label_col], labels)
sess_vals = sorted(raw[sess_col].astype(str).str.strip().unique()) if sess_col else ["NA"]
sess_c = cat(raw[sess_col], sess_vals) if sess_col else np.zeros(len(raw), dtype=np.int32)
assert (subj_c >= 0).all() and (pos_c >= 0).all() and (lab_c >= 0).all()
sig = raw[[chan_cols[k] for k in IMU]].to_numpy(dtype=np.float32)
assert np.isfinite(sig).all(), "non-finite IMU values: handle explicitly before modelling"
n_rows = len(raw); del raw
log("Sessions:", sess_vals)

POS_LIST = POSITIONS or positions_all
assert all(p in positions_all for p in POS_LIST), f"POSITIONS must be among {positions_all}, got {POS_LIST}"

## 4 · Rebuild the windows and verify them against the protocol

Same code as notebooks 01/02. **Every** window (start row, subject, position, session, label, label id, run id) is compared with `protocol/windows_index.parquet`; the LOSO folds are compared with `protocol/loso_folds.json`; the window-start hash is compared with the primary CNN run if it is available. Any difference stops the run.

In [ ]:
W = int(round(WINDOW_SECONDS * FS)); S = max(1, int(round(W * (1 - OVERLAP))))
chg = np.ones(n_rows, dtype=bool)
chg[1:] = ((subj_c[1:] != subj_c[:-1]) | (pos_c[1:] != pos_c[:-1]) |
           (sess_c[1:] != sess_c[:-1]) | (lab_c[1:] != lab_c[:-1]))
starts = np.flatnonzero(chg); lens = np.diff(np.r_[starts, n_rows])
log(f"Runs: {len(starts)} | window = {W} samples, step = {S}")
keep = lens >= W
rs, rl = starts[keep], lens[keep]
nwin = (rl - W) // S + 1
tot = int(nwin.sum())
run_of_win = np.repeat(np.arange(len(rs)), nwin)
offs = np.arange(tot) - np.repeat(np.cumsum(nwin) - nwin, nwin)
w_start = (rs[run_of_win] + offs * S).astype(np.int64)
w_end = w_start + W - 1
for nm, arr in [("subject", subj_c), ("position", pos_c), ("session", sess_c), ("label", lab_c)]:
    assert (arr[w_start] == arr[w_end]).all(), f"window crosses a {nm} boundary"
widx = pd.DataFrame({"start_row": w_start, "subject": [subjects[i] for i in subj_c[w_start]],
                     "position": [positions_all[i] for i in pos_c[w_start]],
                     "session": [sess_vals[i] for i in sess_c[w_start]],
                     "label": [labels[i] for i in lab_c[w_start]], "label_id": lab_c[w_start].astype(np.int16),
                     "run_id": rs[run_of_win]})
WINDOWS_SHA256 = hashlib.sha256(w_start.tobytes()).hexdigest()
log("Windows:", len(widx), "| rows dropped (runs shorter than window):", int(lens[~keep].sum()), "| window-start hash:", WINDOWS_SHA256[:16], "...")

# ---- verify against protocol/windows_index.parquet (required)
p_parq = PROTOCOL_DIR / "windows_index.parquet"
assert p_parq.exists(), f"{p_parq} not found"
ref = pd.read_parquet(p_parq)
assert len(ref) == len(widx), f"window count differs from the protocol: {len(ref)} vs {len(widx)}"
assert (ref["start_row"].to_numpy() == w_start).all(), "window starts differ from the protocol"
assert (ref["label_id"].to_numpy() == widx["label_id"].to_numpy()).all(), "window label ids differ from the protocol"
for c in ["subject", "position", "session", "label"]:
    assert (ref[c].astype(str).to_numpy() == widx[c].astype(str).to_numpy()).all(), f"window {c} differs from the protocol"
assert (ref["run_id"].to_numpy() == widx["run_id"].to_numpy()).all(), "window run ids differ from the protocol"
assert proto_meta["window_samples"] == W and proto_meta["step"] == S and proto_meta["fs"] == FS, "windows_meta.json differs from this run's window settings"
assert proto_meta["labels"] == labels and proto_meta["positions"] == positions_all and proto_meta["subjects"] == subjects, "windows_meta.json differs on labels/positions/subjects"
assert proto_meta["data_sha256"] == DATA_SHA256, "windows_meta.json data hash differs from the data file"
del ref
log("VERIFIED: windows are identical to protocol/windows_index.parquet.")

# ---- primary CNN run used the same windows (optional extra evidence)
_pcnn_dir = find_primary("cnn_full")
if _pcnn_dir is not None and (_pcnn_dir / "run_config.json").exists():
    assert json.load(open(_pcnn_dir / "run_config.json"))["design"]["windows_sha256"] == WINDOWS_SHA256, "window-start hash differs from the primary CNN run"
    log("VERIFIED: window-start hash equals the primary CNN run's.")
else:
    log("Note: primary cnn_full/run_config.json not found under", PRIMARY_DIR, "; window-start hash not compared with the primary CNN run.")

# ---- folds
FOLD_DEFS = [{"fold": i, "test_subject": s, "train_subjects": [x for x in subjects if x != s]} for i, s in enumerate(subjects)]
assert FOLD_DEFS == proto_folds, "LOSO folds differ from protocol/loso_folds.json"
FOLD_LIST = [f for f in FOLD_DEFS if (FOLDS is None or f["fold"] in FOLDS)]
assert FOLD_LIST, "no folds selected"
log("Folds verified against protocol/loso_folds.json | folds this chunk:", [f["fold"] for f in FOLD_LIST])
display(widx.groupby(["position", "label"]).size().unstack("position"))

## 5 · Feature extraction and the leakage unit test

Only per-channel features, nothing mixes axes. The unit test covers all 15 primary configurations: the features of a channel subset must equal the matching columns of the full six-channel matrix, and a k-channel configuration must have exactly 15·k columns.

In [ ]:
FEATS = ["mean", "std", "min", "max", "median", "iqr", "rms", "skew", "kurt", "mad", "zcr",
         "dom_freq", "spec_energy", "spec_entropy", "spec_centroid"]

def window_features(X, ch_names, fs=FS):
    # X: (N, T, C) float -> (N, C*len(FEATS)) float32, names 'Ax__mean', ...
    N, T, C = X.shape
    freqs = np.fft.rfftfreq(T, d=1.0 / fs)[1:]
    out, names = [], []
    for c, ch in enumerate(ch_names):
        x = X[:, :, c].astype(np.float64)
        mu = x.mean(1); sd = x.std(1); xc = x - mu[:, None]
        q25, med, q75 = np.percentile(x, [25, 50, 75], axis=1)
        z = xc / np.where(sd > 1e-12, sd, 1.0)[:, None]
        P = np.abs(np.fft.rfft(xc, axis=1))[:, 1:] ** 2
        tot_p = P.sum(1); p = P / np.where(tot_p > 1e-12, tot_p, 1.0)[:, None]
        cols = [mu, sd, x.min(1), x.max(1), med, q75 - q25, np.sqrt((x ** 2).mean(1)),
                (z ** 3).mean(1), (z ** 4).mean(1) - 3.0, np.abs(np.diff(x, axis=1)).mean(1),
                (np.diff(np.signbit(xc).astype(np.int8), axis=1) != 0).mean(1),
                freqs[P.argmax(1)], tot_p / T, -(p * np.log(p + 1e-12)).sum(1), (p * freqs[None, :]).sum(1)]
        out += cols; names += [f"{ch}__{f}" for f in FEATS]
    return np.stack(out, axis=1).astype(np.float32), names

def gather(rows):                       # raw windows for given start rows -> (n, W, 6)
    return sig[rows[:, None] + np.arange(W)[None, :]]

# ---- unit test over the 15 primary subsets: subset features == matching columns of the full matrix (no cross-channel dependence)
_rng = np.random.default_rng(SEED); _rows = w_start[_rng.choice(len(w_start), size=min(300, len(w_start)), replace=False)]
_Xw = gather(_rows)
_full, FULL_NAMES = window_features(_Xw, IMU)
assert FULL_NAMES == [f"{c}__{f}" for c in IMU for f in FEATS]
for _sub in [list(v) for v in PRIMARY_CONFIGS.values()]:
    _f, _n = window_features(_Xw[:, :, [IMU.index(c) for c in _sub]], _sub)
    _cols = [FULL_NAMES.index(n) for n in _n]
    assert _f.shape[1] == len(FEATS) * len(_sub), f"{_sub}: expected {len(FEATS) * len(_sub)} features, got {_f.shape[1]}"
    assert np.allclose(_f, _full[:, _cols], atol=1e-4, rtol=1e-4), f"feature leakage across channels for {_sub}"
assert np.isfinite(_full).all()
log("Feature unit tests passed for all 15 primary subsets.", len(FEATS), "features per channel.")

def cols_for(ch):
    cols = [i for i, n in enumerate(FULL_NAMES) if n.split("__")[0] in ch]
    assert len(cols) == len(FEATS) * len(ch), f"{ch}: {len(cols)} columns, expected {len(FEATS) * len(ch)}"
    return cols
COLS = {c: cols_for(ch) for c, ch in CONFIGS.items()}

POS_DATA = {}
t0 = time.time()
for p in POS_LIST:
    sel = np.flatnonzero(widx["position"].to_numpy() == p)
    Xp = gather(w_start[sel])                                           # (n, W, 6)
    d = dict(sel=sel, y=widx["label_id"].to_numpy()[sel].astype(np.int64),
             subj=widx["subject"].to_numpy()[sel], starts=w_start[sel])
    d["F"], _nm = window_features(Xp, IMU)
    assert _nm == FULL_NAMES
    POS_DATA[p] = d
    log(f"{p}: windows={len(sel)} | features={d['F'].shape}")
    del Xp
log("Per-position data prepared in", round(time.time() - t0, 1), "s")

## 6 · Models (reused unchanged from notebook 01; the seed is an explicit argument)

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib
if "XGB" in MODELS:
    try:
        import xgboost as xgb
    except ImportError:
        os.system(f"{sys.executable} -m pip install -q xgboost"); import xgboost as xgb

def class_weights(y, K):                 # per-class weights, training windows only (same formula as 01 and 02)
    cnt = np.bincount(y, minlength=K).astype(float)
    return np.where(cnt > 0, len(y) / (K * np.maximum(cnt, 1)), 0.0)

def svm_subsample(ytr, fold, seed):
    # stratified cap of SVM_MAX_TRAIN_WINDOWS training windows (indices into the training set); returns all indices if below the cap
    idx = np.arange(len(ytr))
    if len(idx) > SVM_MAX_TRAIN_WINDOWS:
        idx, _ = train_test_split(idx, train_size=SVM_MAX_TRAIN_WINDOWS, stratify=ytr, random_state=seed + fold)
    return idx

def fit_model(name, Xtr, ytr, fold, seed):
    # returns (model, scaler_or_None). `seed` is the ONLY thing that differs between notebook 01 (seed 42) and this notebook.
    if name == "RF":
        m = RandomForestClassifier(**RF_BASE, random_state=seed).fit(Xtr, ytr)
        return m, None
    if name == "SVM":
        idx = svm_subsample(ytr, fold, seed)
        sc = StandardScaler().fit(Xtr[idx])
        m = SVC(**SVM_BASE, random_state=seed).fit(sc.transform(Xtr[idx]), ytr[idx])
        return m, sc
    if name == "XGB":
        params = dict(XGB_BASE); params["device"] = XGB_DEVICE
        # all K classes are present in training (asserted in the loop), so ids 0..K-1 are contiguous
        m = xgb.XGBClassifier(objective="multi:softprob", random_state=seed, **params)
        m.fit(Xtr, ytr, sample_weight=class_weights(ytr, K)[ytr])
        return m, None
    raise ValueError(name)

def predict(name, m, sc, X):
    Z = sc.transform(X) if sc is not None else X
    return m.predict(Z)

def complexity(name, m, n_feat):
    buf = io.BytesIO(); joblib.dump(m, buf, compress=0); size = buf.tell()
    if name == "RF":
        nodes = int(sum(e.tree_.node_count for e in m.estimators_)); return size, nodes, len(m.estimators_)
    if name == "XGB":
        dump = m.get_booster().get_dump(); return size, int(sum(d.count("\n") for d in dump)), len(dump)
    if name == "SVM":
        nsv = int(m.n_support_.sum()); return size, int(nsv * n_feat + m.dual_coef_.size + m.intercept_.size), nsv

## 7 · Run integrity (config hash) and result schema

The hash covers everything that defines the experiment (windows, features, every model hyper-parameter, data hash, window hash, **this run's seed**, smoke flag). It deliberately **excludes** the chunk selection (`POSITIONS`, `MODELS`, `FOLDS`, `CONFIG_NAMES`, `XGB_DEVICE`), so chunks that are resumed or merged share one hash and results from different settings can never be mixed. The schema equals notebook 01's per-unit columns plus an explicit `seed` column, minus the latency columns.

In [ ]:
design = {"window_samples": W, "step": S, "fs": FS, "features": FEATS, "labels": labels, "positions": positions_all,
          "subjects": subjects, "rf": RF_BASE, "svm": SVM_BASE, "svm_cap": SVM_MAX_TRAIN_WINDOWS, "xgb": XGB_BASE,
          "data_sha256": DATA_SHA256, "windows_sha256": WINDOWS_SHA256, "smoke": SMOKE_TEST, "seed": SEED,
          "configs": {k: PRIMARY_CONFIGS[k] for k in sorted(PRIMARY_CONFIGS)}}
cfg_hash = hashlib.sha256(json.dumps(design, sort_keys=True, default=str).encode()).hexdigest()
cfg_path = RUN_DIR / "run_config.json"
if cfg_path.exists():
    old = json.load(open(cfg_path))
    if old.get("hash") != cfg_hash and not FORCE_NEW_RUN:
        raise RuntimeError("Existing results were produced with DIFFERENT settings. Use a new RESULTS_DIR, or set FORCE_NEW_RUN=1 "
                           "after deleting results.csv yourself.")
json.dump({"hash": cfg_hash, "design": design}, open(cfg_path, "w"), indent=2, default=str)

COLUMNS = ["position", "config", "model", "fold", "seed", "test_subject", "n_channels", "channels", "n_features",
           "n_train_windows", "n_test_windows", "macro_f1", "macro_f1_workouts", "accuracy", "weighted_f1", "macro_precision",
           "macro_recall", "train_time_s", "predict_time_batch_s"] + [f"f1_{l}" for l in labels] + \
          ["model_size_bytes", "n_params_proxy", "n_trees_or_sv", "data_rate_hz"]
RES_PATH = RUN_DIR / "results.csv"

done = set()
if RES_PATH.exists():
    _hdr = pd.read_csv(RES_PATH, nrows=0).columns.tolist()
    assert _hdr == COLUMNS, "existing results.csv has a different column schema than this notebook writes"
    _d = pd.read_csv(RES_PATH, usecols=["position", "config", "model", "fold", "seed"])
    assert (_d["seed"] == SEED).all(), "existing results.csv contains rows of another seed"
    done = set(map(tuple, _d.to_numpy().tolist()))
    assert len(done) == len(_d), "duplicated units already present in results.csv"
log("Units already completed in", RUN_DIR.name, ":", len(done))

# every unit this chunk is responsible for: (position, config, model, fold, seed)
chunk_keys = {(p, c, m, fd["fold"], SEED) for p in POS_LIST for fd in FOLD_LIST for c in CONFIGS for m in MODELS}
total_units = len(chunk_keys)
assert total_units == len(POS_LIST) * len(FOLD_LIST) * len(CONFIGS) * len(MODELS), total_units
log(f"Chunk = {len(POS_LIST)} position(s) x {len(FOLD_LIST)} fold(s) x {len(CONFIGS)} config(s) x models {MODELS} = {total_units} units "
    f"| already done: {len(chunk_keys & done)} | to run: {len(chunk_keys - done)}")
if not SMOKE_TEST and CONFIG_NAMES is None and POSITIONS is None and FOLDS is None and len(MODELS) == 3:
    assert total_units == 15 * 3 * 10 * 3 == 1350, "a full seed (XGB, RF, SVM) should be 1350 units"

## 8 · Sanity checks before the grid

1. **Label-shuffle control for every model in this chunk:** a model trained on shuffled training labels must score near chance (chance ≈ 0.083) on the held-out subject. If it does not, something leaks and the run stops.
2. **The seed really changes the SVM sub-sample:** the stratified 15,000-window sub-sample drawn with this seed is different from the one drawn with seed 42 (otherwise the SVM would not vary with the seed).
3. **Reproduction anchor (full mode only, if notebook 01's results are available):** the 6-axis unit of the first position and fold is retrained here with `seed = 42` and compared with the stored seed-42 macro-F1 of `classical_full` (tolerance 0.001; classical models are deterministic for fixed library versions). A mismatch is reported, not fatal, because it points to a library or hardware difference rather than a protocol change; the anchor file ships with the results.

In [ ]:
SANITY_PATH = RUN_DIR / "sanity_checks.json"
sanity = json.load(open(SANITY_PATH)) if SANITY_PATH.exists() else {}
D0 = POS_DATA[POS_LIST[0]]; fd0 = FOLD_LIST[0]; c0 = next(iter(CONFIGS))
te0 = D0["subj"] == fd0["test_subject"]; tr0 = ~te0
ytr0, yte0 = D0["y"][tr0], D0["y"][te0]; present0 = np.unique(yte0)
y_shuf = np.random.RandomState(SEED).permutation(ytr0)

# ---- 1 · label-shuffle control
for mname in MODELS:
    t0 = time.time()
    m_, sc_ = fit_model(mname, D0["F"][tr0][:, COLS[c0]], y_shuf, fd0["fold"], SEED)
    pred_ = predict(mname, m_, sc_, D0["F"][te0][:, COLS[c0]])
    f1v = float(f1_score(yte0, pred_, labels=present0, average="macro", zero_division=0))
    sanity[mname] = {"label_shuffle_macro_f1": f1v, "chance_reference": 1.0 / K, "position": POS_LIST[0], "fold": fd0["fold"],
                     "config": c0, "threshold": SHUFFLE_THRESHOLD, "seed": SEED}
    log(f"Label-shuffle control {mname} on {POS_LIST[0]} fold {fd0['fold']} config {c0}: macro-F1 = {f1v:.3f} "
        f"(chance ~ {1.0 / K:.3f}) in {time.time() - t0:.1f}s")
    assert f1v < SHUFFLE_THRESHOLD, f"{mname}: shuffled-label model is far above chance -> LEAKAGE or a bug. Do not trust any result."
    del m_, sc_

# ---- 2 · the seed changes the SVM sub-sample
if "SVM" in MODELS:
    a42 = np.sort(svm_subsample(ytr0, fd0["fold"], SEED_PRIMARY)); aNew = np.sort(svm_subsample(ytr0, fd0["fold"], SEED))
    if len(ytr0) > SVM_MAX_TRAIN_WINDOWS:
        assert not np.array_equal(a42, aNew), "the SVM sub-sample is identical for seed 42 and this seed"
        ov = len(np.intersect1d(a42, aNew)) / len(a42)
        sanity["svm_subsample_overlap_with_seed42"] = float(ov)
        log(f"Check 2 passed: SVM sub-sample differs from seed 42's (overlap {ov:.3f} of {len(a42)} windows).")
    else:
        log("Check 2 skipped: training set is not above the SVM cap.")
json.dump(sanity, open(SANITY_PATH, "w"), indent=2)

# ---- 3 · reproduction anchor (full mode only)
ANCHOR_PATH = RUN_DIR / "anchor_check.csv"
_pdir = find_primary("classical_full")
def primary_f1(mname, p, fold):
    if _pdir is None or not (_pdir / "results.csv").exists(): return None
    d = pd.read_csv(_pdir / "results.csv")
    q = d[(d["position"] == p) & (d["config"] == "6-axis") & (d["model"] == mname) & (d["fold"] == fold)]
    if "seed" in q.columns: q = q[q["seed"] == SEED_PRIMARY]
    return float(q["macro_f1"].iloc[0]) if len(q) == 1 else None

if SMOKE_TEST:
    log("Reproduction anchor skipped in smoke mode (reduced model sizes cannot reproduce the primary run).")
else:
    pA, fA = POS_LIST[0], FOLD_LIST[0]
    have = set()
    if ANCHOR_PATH.exists():
        _a = pd.read_csv(ANCHOR_PATH); have = set(zip(_a["model"], _a["position"], _a["fold"]))
    for mname in MODELS:
        if (mname, pA, fA["fold"]) in have: continue
        ref_f1 = primary_f1(mname, pA, fA["fold"])
        if ref_f1 is None:
            log(f"Anchor {mname}: primary result not found under {PRIMARY_DIR}; skipped."); continue
        D = POS_DATA[pA]; te_a = D["subj"] == fA["test_subject"]; tr_a = ~te_a
        ytr_a, yte_a = D["y"][tr_a], D["y"][te_a]; pres_a = np.unique(yte_a)
        t0 = time.time()
        mod_a, sc_a = fit_model(mname, D["F"][tr_a], ytr_a, fA["fold"], SEED_PRIMARY)       # seed 42 on purpose
        pred_a = predict(mname, mod_a, sc_a, D["F"][te_a])
        new_f1 = float(f1_score(yte_a, pred_a, labels=pres_a, average="macro", zero_division=0))
        ok = abs(new_f1 - ref_f1) <= ANCHOR_TOL
        pd.DataFrame([dict(model=mname, position=pA, fold=fA["fold"], config="6-axis", seed=SEED_PRIMARY, macro_f1_here=new_f1,
                           macro_f1_primary=ref_f1, abs_diff=abs(new_f1 - ref_f1), tolerance=ANCHOR_TOL, within_tolerance=bool(ok),
                           seconds=time.time() - t0)]
                     ).to_csv(ANCHOR_PATH, mode="a", header=not ANCHOR_PATH.exists(), index=False)
        log(f"Anchor {mname} 6-axis {pA} fold {fA['fold']} (seed 42): here {new_f1:.4f} vs primary {ref_f1:.4f} (|diff| {abs(new_f1 - ref_f1):.4f}, "
            f"tolerance {ANCHOR_TOL}) -> {'OK' if ok else 'MISMATCH: check library versions / hardware before comparing seeds with the primary run'}")
        del mod_a, sc_a

## 9 · Main LOSO loop (resumable, chunkable)

Loop order: position → fold → configuration → model (as in notebook 01). Each finished unit is appended to `results.csv`; finished units are skipped. If `TIME_BUDGET_MIN` is set, the loop stops cleanly before starting a unit once the budget is used up, and everything is written and zipped.

In [ ]:
workout_ids = [i for i in range(K) if i not in null_ids]
rows_buf = []

class StopRun(Exception):
    pass

def check_time():
    if TIME_BUDGET_S is not None and time.time() - T0 > TIME_BUDGET_S:
        raise StopRun()

def flush():
    global rows_buf
    if rows_buf:
        pd.DataFrame(rows_buf).reindex(columns=COLUMNS).to_csv(RES_PATH, mode="a", header=not RES_PATH.exists(), index=False); rows_buf = []

def make_row(p, cname, mname, fd, ch, yte, pred, present, n_features, n_tr, n_te, t_fit, t_pred):
    f1c = f1_score(yte, pred, labels=np.arange(K), average=None, zero_division=0)
    pw = [i for i in workout_ids if i in present]
    row = dict(position=p, config=cname, model=mname, fold=fd["fold"], seed=SEED, test_subject=fd["test_subject"],
               n_channels=len(ch), channels="".join(ch), n_features=n_features, n_train_windows=n_tr, n_test_windows=n_te,
               macro_f1=float(f1_score(yte, pred, labels=present, average="macro", zero_division=0)),
               macro_f1_workouts=float(f1c[pw].mean()) if pw else np.nan,
               accuracy=float(accuracy_score(yte, pred)),
               weighted_f1=float(f1_score(yte, pred, labels=present, average="weighted", zero_division=0)),
               macro_precision=float(precision_score(yte, pred, labels=present, average="macro", zero_division=0)),
               macro_recall=float(recall_score(yte, pred, labels=present, average="macro", zero_division=0)),
               train_time_s=t_fit, predict_time_batch_s=t_pred, data_rate_hz=len(ch) * FS)
    for i, l in enumerate(labels): row[f"f1_{l}"] = float(f1c[i]) if i in present else np.nan
    return row

def run_classical(p, fd, cname, ch, mname, Ftr_all, Fte_all, ytr, yte, present, n_tr, n_te):
    cols = COLS[cname]
    Xtr, Xte = Ftr_all[:, cols], Fte_all[:, cols]
    t0 = time.time(); model, sc = fit_model(mname, Xtr, ytr, fd["fold"], SEED); t_fit = time.time() - t0
    t0 = time.time(); pred = predict(mname, model, sc, Xte); t_pred = time.time() - t0
    row = make_row(p, cname, mname, fd, ch, yte, pred, present, len(cols), n_tr, n_te, t_fit, t_pred)
    size, n_par, n_units = complexity(mname, model, len(cols))
    row.update(model_size_bytes=size, n_params_proxy=n_par, n_trees_or_sv=n_units)
    return row

n_new, t_start, stopped_early = 0, time.time(), False
try:
    for p in POS_LIST:
        D = POS_DATA[p]
        for fd in FOLD_LIST:
            if not any(k[0] == p and k[3] == fd["fold"] and k not in done for k in chunk_keys):
                continue
            test_s = fd["test_subject"]; te = D["subj"] == test_s; tr = ~te
            # ---- leakage guards
            assert test_s not in set(D["subj"][tr]), "test subject present in training windows"
            assert set(D["subj"][tr]) == set(fd["train_subjects"]), "training subjects differ from fold definition"
            ytr, yte = D["y"][tr], D["y"][te]
            assert len(np.unique(ytr)) == K, "a class is missing from the training subjects"
            present = np.unique(yte)
            n_tr, n_te = int(tr.sum()), int(te.sum())
            Ftr_all, Fte_all = D["F"][tr], D["F"][te]
            for cname, ch in CONFIGS.items():
                for mname in MODELS:
                    key = (p, cname, mname, fd["fold"], SEED)
                    if key in done: continue
                    check_time()
                    row = run_classical(p, fd, cname, ch, mname, Ftr_all, Fte_all, ytr, yte, present, n_tr, n_te)
                    rows_buf.append(row); done.add(key); n_new += 1
                    if len(rows_buf) >= 10: flush()
                    if n_new % 25 == 0 or n_new == 1:
                        el = time.time() - t_start; rem = len(chunk_keys - done)
                        log(f"[{len(chunk_keys & done)}/{total_units}] {p} fold={fd['fold']} {cname} {mname}: macroF1={row['macro_f1']:.3f} "
                            f"fit={row['train_time_s']:.1f}s | elapsed {el / 60:.1f} min, ETA ~ {el / n_new * rem / 60:.1f} min")
            flush()
except StopRun:
    stopped_early = True
    log(f"TIME BUDGET reached ({TIME_BUDGET_S / 60:.1f} min): stopping cleanly. Re-run (or resume) to finish the remaining units.")
finally:
    flush()
log("Loop finished. New units this session:", n_new, "| chunk units done:", len(chunk_keys & done), "of", total_units,
    "| stopped early:", stopped_early)

## 10 · Completeness check and descriptive comparison with seed 42

Descriptive only; the seed-variability tables and the claim re-checks are produced by notebook `04c_seed_analysis`. If notebook 01's results are available, the table below shows, per (position, model, configuration), the mean macro-F1 over subjects for seed 42 and for this seed, and the paired per-subject difference (mean and SD).

In [ ]:
assert RES_PATH.exists(), "No unit has finished yet, so there is no results.csv (time budget too small, or nothing was selected). Re-run with a larger TIME_BUDGET_MIN."
res = pd.read_csv(RES_PATH)
dups = res.duplicated(subset=["position", "config", "model", "fold", "seed"]).sum()
assert dups == 0, f"{dups} duplicated units in results.csv"
assert (res["seed"] == SEED).all(), "results.csv contains rows of another seed"
assert res["config"].isin(list(PRIMARY_CONFIGS)).all(), "results.csv contains a configuration outside the primary 15"
assert (res["n_channels"] == res["config"].map(lambda c: len(PRIMARY_CONFIGS[c]))).all(), "n_channels inconsistent with the configuration"
assert (res["channels"] == res["config"].map(lambda c: "".join(PRIMARY_CONFIGS[c]))).all(), "channel string inconsistent with the configuration"
assert (res["n_features"] == len(FEATS) * res["n_channels"]).all(), "n_features is not 15 x n_channels"

res_keys = set(zip(res["position"], res["config"], res["model"], res["fold"], res["seed"]))
missing = sorted(chunk_keys - res_keys)
complete = len(missing) == 0
log("Rows in results.csv:", len(res), "| chunk units present:", len(chunk_keys & res_keys), "of", total_units,
    "| COMPLETE" if complete else f"| INCOMPLETE ({len(missing)} missing): re-run to finish")
json.dump({"seed": SEED, "chunk_units": total_units, "present": len(chunk_keys & res_keys), "missing": len(missing), "complete": complete,
           "stopped_early": stopped_early, "rows_in_results_csv": int(len(res)),
           "missing_examples": [list(map(str, k)) for k in missing[:20]]}, open(RUN_DIR / "completeness.json", "w"), indent=2)
if REQUIRE_COMPLETE:
    assert complete, f"REQUIRE_COMPLETE=1 but {len(missing)} units of this chunk are missing"

summ = (res.groupby(["position", "model", "config"])
        .agg(n_folds=("fold", "count"), n_channels=("n_channels", "first"), macro_f1_mean=("macro_f1", "mean"), macro_f1_sd=("macro_f1", "std"),
             macro_f1_workouts_mean=("macro_f1_workouts", "mean"), accuracy_mean=("accuracy", "mean"),
             weighted_f1_mean=("weighted_f1", "mean"), train_time_s_mean=("train_time_s", "mean"))
        .reset_index())
summ["seed"] = SEED
summ.to_csv(RUN_DIR / "summary_by_position_model_config.csv", index=False)

# ---- descriptive comparison with the seed-42 run of notebook 01 (full mode only)
if SMOKE_TEST:
    log("Seed-vs-primary comparison skipped in smoke mode (reduced model sizes).")
elif _pdir is None or not (_pdir / "results.csv").exists():
    log("Seed-vs-primary comparison skipped: classical_full/results.csv not found under", PRIMARY_DIR)
else:
    prim = pd.read_csv(_pdir / "results.csv")
    if "seed" in prim.columns: prim = prim[prim["seed"] == SEED_PRIMARY]
    prim = prim[["position", "config", "model", "fold", "macro_f1"]].rename(columns={"macro_f1": "macro_f1_seed42"})
    cmp_ = res[["position", "config", "model", "fold", "macro_f1"]].rename(columns={"macro_f1": "macro_f1_this_seed"}).merge(
        prim, on=["position", "config", "model", "fold"], how="inner")
    cmp_["diff"] = cmp_["macro_f1_this_seed"] - cmp_["macro_f1_seed42"]
    sv = (cmp_.groupby(["position", "model", "config"])
          .agg(n_folds=("fold", "count"), macro_f1_seed42=("macro_f1_seed42", "mean"), macro_f1_this_seed=("macro_f1_this_seed", "mean"),
               mean_diff=("diff", "mean"), sd_diff_across_subjects=("diff", "std"), max_abs_subject_diff=("diff", lambda x: float(np.abs(x).max())))
          .reset_index())
    sv["seed"] = SEED
    sv.to_csv(RUN_DIR / "seed_vs_primary.csv", index=False)
    log("Seed", SEED, "vs seed 42 over", len(cmp_), "matched units | mean |cell-level diff|:", round(float(sv["mean_diff"].abs().mean()), 4),
        "| max |cell-level diff|:", round(float(sv["mean_diff"].abs().max()), 4))
    display(sv.round(4).head(20))
display(summ.round(4).head(20))

## 11 · Environment record, session log and zip

In [ ]:
import importlib
def ver(m):
    try: return importlib.import_module(m).__version__
    except Exception: return "n/a"
def cpu_name():
    try:
        for line in open("/proc/cpuinfo"):
            if line.lower().startswith("model name"): return line.split(":", 1)[1].strip()
    except Exception:
        pass
    return platform.processor() or "n/a"

chunk_spec = {"positions": POS_LIST, "models": MODELS, "configs": list(CONFIGS), "folds": [f["fold"] for f in FOLD_LIST],
              "seed": SEED, "smoke_test": SMOKE_TEST, "xgb_device": XGB_DEVICE, "time_budget_min": None if TIME_BUDGET_S is None else TIME_BUDGET_S / 60}
env = {"python": sys.version, "platform": platform.platform(), "cpu": cpu_name(), "cpu_count": os.cpu_count(),
       "seed": SEED, "primary_seed": SEED_PRIMARY, "chunk": chunk_spec,
       "packages": {m: ver(m) for m in ["numpy", "pandas", "sklearn", "xgboost", "joblib", "pyarrow"]},
       "data_sha256": DATA_SHA256, "windows_sha256": WINDOWS_SHA256, "config_hash": cfg_hash,
       "windows_verified_against_protocol": True, "folds_verified_against_protocol": True,
       "complete_chunk": complete, "timestamp_utc": pd.Timestamp.now("UTC").isoformat()}
json.dump(env, open(RUN_DIR / "environment.json", "w"), indent=2)
with open(RUN_DIR / "sessions.jsonl", "a") as f:
    f.write(json.dumps({"timestamp_utc": env["timestamp_utc"], "chunk": chunk_spec, "new_units": n_new, "complete_chunk": complete,
                        "cpu": env["cpu"], "packages": env["packages"]}) + "\n")
with open(RUN_DIR / "run_log.txt", "a") as f:
    f.write(f"\n===== session {env['timestamp_utc']} | chunk {json.dumps(chunk_spec)} =====\n" + "\n".join(LOG) + "\n")

_chunk_tag = "-".join(POS_LIST) + "_" + "-".join(MODELS)
zip_path = BASE / f"recgym_04b_classical_seed{SEED}_{TAG}_{_chunk_tag}.zip"
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in RUN_DIR.rglob("*"):
        if f.is_file(): z.write(f, f"{RUN_DIR.name}/{f.relative_to(RUN_DIR)}")
log("ZIP:", zip_path, "| MB:", round(os.path.getsize(zip_path) / 1e6, 2))
if SMOKE_TEST:
    log("Smoke run finished. Check: the label-shuffle controls, the SVM sub-sample check, the unit counts and the timing/ETA above. Then set SMOKE_TEST=0.")
elif not complete:
    log("This chunk is INCOMPLETE. Re-run with the same settings (or attach this zip's folder via RESUME_FROM) to finish it.")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    log("Not on Colab: find the zip in your working directory / Kaggle Output tab.")